# Ionic conductivity of aqueous sodium acetate (3/3): analysis with STACIE

This notebook analyzes the production run of `02_nvt_production.ipynb`:

1. A few sanity checks of the production run.
2. A visualization of the trajectory.
3. The computation of the charge current from the ion positions.
4. The estimation of the ionic conductivity and its uncertainty with
   [STACIE](https://molmod.github.io/stacie/).

**Theory.**
The ionic electrical conductivity $\sigma$ follows from the Green-Kubo relation:

$$
\sigma = \frac{1}{V k_\text{B} T} \int_0^\infty \bigl\langle J_x(0)\, J_x(t) \bigr\rangle \,\mathrm{d}t
$$

where $V$ is the volume, $k_\text{B}$ the Boltzmann constant, $T$ the temperature
and $J_x$ the $x$-component of the charge current:

$$
\mathbf{J}(t) = \sum_{n} q_n \mathbf{v}_n(t)
$$

The sum runs over all ions, with charges $q_n$ and (center of mass) velocities $\mathbf{v}_n$.
(Water molecules are neutral and do not contribute to the conductivity.)
In practice, the integral of the autocorrelation function is difficult to compute directly,
because the autocorrelation function becomes very noisy at long times.
STACIE avoids this problem by fitting a model to the low-frequency part of the power spectrum of the charge current,
whose value at zero frequency is the autocorrelation integral.
It also provides an estimate of the uncertainty of the result.
See the [STACIE documentation](https://molmod.github.io/stacie/) for more details.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np
import openmm as mm
import pandas
import scipy.constants as sc
from stacie import (
    DebyeModel,
    UnitConfig,
    compute_spectrum,
    estimate_acint,
    plot_extras,
    plot_fitted_spectrum,
)

In [ ]:
# Settings of the production run (see 02_nvt_production.ipynb)
temperature = 293.15  # K
# Time between two frames in traj_ions.dcd,
# i.e. interval_ions * timestep in 02_nvt_production.ipynb.
frame_interval = 0.1  # ps
# Number of blocks into which the production run is split for the analysis.
nblock = 10
# Experimental conductivity of 10 mass% sodium acetate at 20 °C (CRC Handbook).
# See the introduction of 01_npt_equilibration.ipynb for the source.
conductivity_exp = 5.34  # S/m

## 1. Sanity checks of the production run

The production run is only useful if the thermostat produces the right temperature distribution
and if the conserved quantity of the Nosé-Hoover integrator does not drift.

In the canonical ensemble, the instantaneous temperature fluctuates around its average
with a standard deviation of $T\sqrt{2/N_\text{dof}}$,
where $N_\text{dof}$ is the number of degrees of freedom:
three per atom, minus the number of constraints,
minus three for the conserved total momentum.

In [ ]:
df = pandas.read_csv("scalars_nvt.csv")
df = df.rename(columns={'#"Progress (%)"': "Progress (%)"})
plt.close("sanity")
fig, axs = plt.subplots(2, 1, sharex=True, figsize=(6, 5), num="sanity")
df.plot(x="Time (ps)", y="Temperature (K)", ax=axs[0], legend=False)
axs[0].set_ylabel("Temperature (K)")
df.plot(x="Time (ps)", y="Conserved Energy (kJ/mole)", ax=axs[1], legend=False)
axs[1].set_ylabel("Conserved energy (kJ/mol)")
fig.tight_layout()

In [ ]:
# Number of degrees of freedom
with open("init_system.xml") as f:
    system = mm.XmlSerializer.deserialize(f.read())
ndof = 3 * system.getNumParticles() - system.getNumConstraints() - 3
print(f"Number of degrees of freedom: {ndof}")

# Temperature distribution
temps = df["Temperature (K)"]
print(f"Average temperature: {temps.mean():.2f} K (expected {temperature:.2f} K)")
print(
    f"Std of the temperature: {temps.std():.2f} K "
    f"(expected {temperature * np.sqrt(2 / ndof):.2f} K)"
)

# Drift of the conserved quantity, expressed in units of kT per degree of freedom per ns.
slope = np.polyfit(df["Time (ps)"], df["Conserved Energy (kJ/mole)"], 1)[0]  # kJ/mol/ps
kt = sc.k * sc.N_A * temperature / 1000  # kJ/mol
print(f"Drift of the conserved quantity: {slope * 1000 / (ndof * kt):.2e} kT per dof per ns")

# The speed in the first row is zero, because it is not measured yet.
speed = df["Speed (ns/day)"].iloc[1:].mean()
print(f"Average speed: {speed:.1f} ns/day")

## 2. Visualization

The snapshots in `traj_nvt.dcd` are wrapped into the simulation cell.
Sodium ions are shown as spheres and acetate ions as balls and sticks.
For clarity, the water molecules are not shown.

In [ ]:
traj = mdtraj.load("traj_nvt.dcd", top="final.pdb")
view = nglview.show_mdtraj(traj)
view.clear_representations()
view.add_ball_and_stick("ACT")
view.add_spacefill("NA", radius=0.6)
view.add_unitcell()
view

## 3. Charge current

The file `traj_ions.dcd` contains the positions of one atom per ion every 100 fs.
The charge current is computed from the change in the dipole moment $\mathbf{M}$ of the ions
between two consecutive frames:

$$
\mathbf{M}(t) = \sum_n q_n \mathbf{r}_n(t)
\qquad
\bar{\mathbf{J}}(t + \Delta t/2) = \frac{\mathbf{M}(t + \Delta t) - \mathbf{M}(t)}{\Delta t}
$$

This finite difference is exactly the average of the charge current over the time interval $\Delta t$.
(Such a "block average" is a good input for STACIE,
as long as the interval is short compared to the relevant timescales of the charge current.)

The trajectory is loaded one block of 1 ns at a time, to limit the memory usage.
Each block is treated as an independent sample by STACIE.
This is justified when the autocorrelation function of the charge current
decays much faster than the block length,
which is the case for aqueous electrolytes (picoseconds versus 1 ns).

In [ ]:
# Topology of the ions in traj_ions.dcd: the same selection as in 02_nvt_production.ipynb
top = mdtraj.load_topology("final.pdb")
ion_indices = top.select("resname NA or (resname ACT and name C2)")
top_ions = top.subset(ion_indices)
charges = np.array([1.0 if atom.residue.name == "NA" else -1.0 for atom in top_ions.atoms])
print("Number of ions:", top_ions.n_atoms, " Net charge:", charges.sum())

# Number of frames per block
with mdtraj.formats.DCDTrajectoryFile("traj_ions.dcd") as f:
    nframe = len(f)
nframe_block = nframe // nblock
print(f"{nblock} blocks of {nframe_block} frames ({nframe_block * frame_interval:g} ps)")

In [ ]:
currents = []
for traj_block in mdtraj.iterload("traj_ions.dcd", top=top_ions, chunk=nframe_block):
    if len(traj_block) < nframe_block:
        # Skip the remaining frames that do not fill a complete block.
        break
    # Dipole moment of the ions in e·nm, shape (nframe_block, 3)
    dipole = np.einsum("fai,a->fi", traj_block.xyz, charges)
    # Charge current in e·nm/ps, shape (3, nframe_block - 1)
    currents.append(np.diff(dipole, axis=0).T / frame_interval)
    # The volume is constant in the NVT ensemble.
    volume = traj_block.unitcell_volumes[0]  # nm³
print("Number of blocks:", len(currents))
print(f"Volume: {volume:.3f} nm³")

In [ ]:
# Plot the components of the charge current during the first 20 ps.
plt.close("current")
fig, ax = plt.subplots(num="current")
nplot = min(200, currents[0].shape[1])
time = np.arange(nplot) * frame_interval
ax.plot(time, currents[0][0, :nplot])
ax.plot(time, currents[0][1, :nplot])
ax.plot(time, currents[0][2, :nplot])
ax.set_xlabel("Time (ps)")
ax.set_ylabel("Charge currents $J_x$, $J_y$, $J_z$ (e·nm/ps)")

## 4. Conductivity with STACIE

All quantities are converted to SI units, such that the result is obtained in S/m.
The three Cartesian components of the 10 blocks give 30 sequences,
which STACIE treats as independent samples of the same stochastic process.
The prefactor $1/(V k_\text{B} T)$ is the same for all sequences.

The spectrum of the charge current has a minimum at zero frequency,
whose value is the conductivity.
At higher frequencies, the spectrum increases,
because the ions are temporarily trapped by their neighbors and oscillate in place,
while on longer timescales they escape and diffuse.
The `DebyeModel` describes the onset of this rise with three parameters:
the zero-frequency limit, the height of the rise and a relaxation time.

In [ ]:
# Convert the charge current from e·nm/ps to C·m/s
current_unit = sc.value("elementary charge") * 1e-9 / 1e-12
sequences = np.concatenate(currents) * current_unit
print("Shape of the input for STACIE:", sequences.shape)

spectrum = compute_spectrum(
    sequences,
    prefactors=1 / (volume * 1e-27 * sc.k * temperature),
    timestep=frame_interval * 1e-12,
    include_zero_freq=False,
)

uc = UnitConfig(
    acint_symbol=r"\sigma",
    acint_unit_str="S/m",
    acint_fmt=".2f",
    time_unit=1e-12,
    time_unit_str="ps",
    time_fmt=".3f",
    freq_unit=1e12,
    freq_unit_str="THz",
)
result = estimate_acint(spectrum, DebyeModel(), verbose=True, uc=uc)

In [ ]:
plt.close("spectrum")
fig, ax = plt.subplots(num="spectrum")
plot_fitted_spectrum(ax, uc, result)

plt.close("extras")
fig, axs = plt.subplots(2, 2, num="extras", figsize=(8, 6))
plot_extras(axs, uc, result)
fig.tight_layout()

In [ ]:
print(f"Simulation: {result.acint:.2f} ± {result.acint_std:.2f} S/m")
print(f"Experiment: {conductivity_exp:.2f} S/m")

## 5. How reliable is this result?

STACIE fits the model to the spectrum below a cutoff frequency,
which it selects automatically by comparing many candidate cutoffs.
The screen output and the figures above contain several indicators of the quality of the result:

- **Relative error.**
  The uncertainty of the conductivity is derived from the statistical noise in the spectrum.
  As a rough guideline, the relative error is about $1/\sqrt{M N_\text{eff}}$,
  where $M$ is the number of independent sequences (here 30)
  and $N_\text{eff}$ the effective number of fitted points.
  Longer simulations reduce the error, but only slowly: four times more data halves the error.
- **Effective number of points.**
  This should be at least 20 times the number of model parameters, i.e. at least 60 for the Debye model.
  Fewer points mean that the fit is based on too little information,
  and that the error estimate itself becomes unreliable.
- **Z-scores of the regression cost and the cutoff criterion.**
  These test whether the model is consistent with the data.
  Values larger than about 2 indicate that the model does not describe the spectrum well,
  or that there is not enough data.
- **Cutoff weight** (top left in the extra figures).
  STACIE averages over fits with different cutoff frequencies.
  If the weight is large at the lowest cutoff frequencies,
  the relevant features of the spectrum are not resolved by the frequency grid,
  and longer blocks (i.e. longer simulations) are needed.
  Conversely, if its maximum lies at the highest cutoff frequency,
  first check whether the screen output above reports
  "Reached the maximum number of effective points".
  If so, the scan over the cutoff frequencies was stopped by the `neff_max` argument
  of `estimate_acint` (default 1000),
  and you can rerun the analysis with a larger value, e.g. `neff_max=5000`.
  If not, the spectrum is not well resolved up to sufficiently high frequencies,
  and a higher time resolution (smaller `interval_ions` in the production run)
  could produce a more accurate result.
- **Relaxation time** $\tau_\text{exp}$ of the Debye model.
  This should be much shorter than the block length (1 ns),
  which justifies the treatment of the blocks as independent samples.

**Comparison with experiment.**
Even when the statistical uncertainty is small, the result may deviate considerably from the experimental value,
due to approximations in the force field:

- The CHARMM TIP3P water model diffuses about twice as fast as real water,
  which tends to overestimate the mobility of the ions.
- Non-polarizable models with integer charges overestimate the attraction between ions,
  which leads to too much ion pairing and an underestimation of the conductivity.
  (The NBFIX corrections in CHARMM36 partially address this for Na⁺ and acetate.)
- Finite-size effects: the box is only a few nanometers wide.

**<span style="color:#A03;font-size:14pt">
&#x1F450; HANDS-ON! &#x1F528;
</span>**

> - Rerun the analysis with fewer blocks (e.g. `nblock = 5`) or with only a part of the data,
>   to see how the uncertainty depends on the amount of data.
> - Repeat the three notebooks for sodium chloride.
>   You need a template PDB file for Cl⁻ (similar to `sodium.pdb`),
>   and you must adapt the molar mass of the salt and the selection of the ions.
>   The CRC Handbook reports a conductivity of 126 mS/cm for 10 mass% NaCl at 20 °C.